In [2]:
import geopandas as gpd

import pandas as pd

In [3]:
# config
pd.set_option('display.max_colwidth', None)

gdf = gpd.read_file("data/historic_airphoto_index/main_airphoto_index.gpkg")

## Calculating extents from center points

In match_multi_zoom, the plugin ultimately calls:
tile_fetcher.fetch_tiles_from_center(
    center_lat,
    center_lon,
    extent_meters,
    extent_dimension="horizontal",
    source_aspect_ratio,
    basemap_source,
    zoom_level
)

The function definition tells us exactly what extent_meters represents:
extent_meters = width (horizontal) OR height (vertical) of the basemap area, in meters [mn365-my.s...epoint.com]
…which is then converted into a rectangle in EPSG:3857:
half_width = width_meters / 2
half_height = height_meters / 2

extent = QgsRectangle(
    center_3857.x() - half_width,
    center_3857.y() - half_height,
    center_3857.x() + half_width,
    center_3857.y() + half_height
)

So the extent_meters must be a real-world width (or height) of your historic air photo footprint.

We just need 1 dimension - probably the height? 

We can get the dimensions and DPI of the photo from the JPEG, but in order to know its real world size, we'll need its scale. 

### How much variation in scale is there due to different flight series? 

https://www.dnr.state.mn.us/maps/historical_airphotos/index.html

How feasible is it for me to guess at the scale? How would I divide this into chunks, if needed? 

https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery - detailed info on imagery ordered by county


In [4]:
# get a list of project codes, which loosely correspond to counties

unique_projects = gdf['project'].unique().tolist()
unique_projects

['ait',
 'bec',
 'bel',
 'bhm',
 'bio',
 'bip',
 'biq',
 'bir',
 'bis',
 'bit',
 'biu',
 'biv',
 'biw',
 'bix',
 'biy',
 'biz',
 'bja',
 'bjb',
 'bjc',
 'bjd',
 'bje',
 'bjf',
 'bjg',
 'bjh',
 'bji',
 'bjj',
 'bjk',
 'bjl',
 'bjm',
 'bjn',
 'bjo',
 'bjp',
 'bjq',
 'bjr',
 'bjs',
 'blh',
 'bse',
 'bxn',
 'bxo',
 'bxq',
 'bxr',
 'bxs',
 'bxt',
 'bxu',
 'bxv',
 'bxw',
 'bxx',
 'bxy',
 'bxz',
 'bya',
 'byb',
 'byc',
 'byd',
 'bye',
 'byf',
 'byg',
 'byh',
 'byi',
 'byj',
 'ca',
 'cas',
 'cass',
 'cin',
 'ciq',
 'cir',
 'cis',
 'cit',
 'coo',
 'crow',
 'cs',
 'cwc',
 'czm',
 'da',
 'dch',
 'det',
 'dg',
 'dv',
 'eui',
 'hub',
 'ita',
 'jo',
 'jq',
 'koo',
 'napp',
 'od',
 'p',
 'sb',
 'sd',
 'slc',
 'stl',
 'vbj',
 'vey',
 'vib',
 'vse',
 'vvn',
 'vx',
 'vy',
 'wa_d',
 'wd',
 'we',
 'wf',
 'wg',
 'wl',
 'wm',
 'wn',
 'wp']

In [5]:
year_values = gdf['year'].unique().tolist()
print(year_values)

[1975, 1982, 1973, 1947, 1949, 1976, 1939, 1938, 1955, 1964, 1961, 1950, 1951, 1965, 1954, 1953, 1963, 1962, 1968, 1940, 1960, 1966, 1979, 1969, 1941, 1934, 2002, 2007, 2012, 1972, 1936, 1959, 1978, 1989, 1980, 1988, 1991, 1952, 1948, 1981, 1956, 1958, 1937, 1957]


In [ ]:
# how many different categories are these photos in? 
provider_values = gdf['provider'].nunique()
print(provider_values)

provider_counts = gdf['provider'].value_counts(dropna=False)
print(provider_counts)

print("---------")

project_values = gdf['project'].nunique()
print(project_values)

project_counts = gdf['project'].value_counts(dropna=False)
print(project_counts)

print("---------")
year_values = gdf['year'].nunique()
print(year_values)

year_counts = gdf['year'].value_counts(dropna=False)
print(year_counts)

In [ ]:
gdf

,OBJECTID_1,filename,provider,project,year,photo_id,ext,xutm,yutm,status,image_url,globalid,geometry,download_filename,output_filename
0,17660,/usda/bjf/y1938/bjf02002.jpg,usda,bjf,1938,bjf02002,jpg,260432.0,4868321.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02002.jpg,{6C21141B-1856-46E2-B199-FB9678486151},POINT (260431.708 4868321.904),usda_bjf_y1938_bjf02002.jpg,usda_bjf_y1938_bjf02002_georef.tif
1,17662,/usda/bjf/y1938/bjf02004.jpg,usda,bjf,1938,bjf02004,jpg,257333.0,4868367.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02004.jpg,{967F845D-5444-40B6-BF05-91437D1CAF63},POINT (257332.011 4868368.265),usda_bjf_y1938_bjf02004.jpg,usda_bjf_y1938_bjf02004_georef.tif
2,17664,/usda/bjf/y1938/bjf02006.jpg,usda,bjf,1938,bjf02006,jpg,254233.0,4868414.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02006.jpg,{04683AF5-AE59-4778-AC4B-849119DEC0B0},POINT (254232.313 4868414.627),usda_bjf_y1938_bjf02006.jpg,usda_bjf_y1938_bjf02006_georef.tif
3,17665,/usda/bjf/y1938/bjf02012.jpg,usda,bjf,1938,bjf02012,jpg,253965.0,4865651.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02012.jpg,{8245AC47-1BED-42D5-81E2-79CDEF7E44BF},POINT (253964.528 4865651.395),usda_bjf_y1938_bjf02012.jpg,usda_bjf_y1938_bjf02012_georef.tif
4,17667,/usda/bjf/y1938/bjf02014.jpg,usda,bjf,1938,bjf02014,jpg,256931.0,4865569.0,1,https://maps.dnr.state.mn.us/airphotos/usda/bjf/y1938/bjf02014.jpg,{4F0BDE31-A7AA-4781-A669-29CF8E7C3DE2},POINT (256930.424 4865569.669),usda_bjf_y1938_bjf02014.jpg,usda_bjf_y1938_bjf02014_georef.tif
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93756,96459,/usda/byg/y1966/byg05gg117.jpg,usda,byg,1966,byg05gg117,jpg,319917.0,5429225.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg117.jpg,{FFE59653-BF0D-4BFE-AA97-F5B512DEAA4B},POINT (319916.86 5429226.027),usda_byg_y1966_byg05gg117.jpg,usda_byg_y1966_byg05gg117_georef.tif
93757,96460,/usda/byg/y1966/byg05gg119.jpg,usda,byg,1966,byg05gg119,jpg,323196.0,5429165.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg119.jpg,{FB5FE609-92DD-4B07-AC46-624C3DB7959D},POINT (323195.16 5429166.422),usda_byg_y1966_byg05gg119.jpg,usda_byg_y1966_byg05gg119_georef.tif
93758,96461,/usda/byg/y1966/byg05gg121.jpg,usda,byg,1966,byg05gg121,jpg,326395.0,5429046.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg121.jpg,{2933DB1E-682F-464A-99F8-369190014377},POINT (326393.986 5429047.211),usda_byg_y1966_byg05gg121.jpg,usda_byg_y1966_byg05gg121_georef.tif
93759,96462,/usda/byg/y1966/byg05gg123.jpg,usda,byg,1966,byg05gg123,jpg,329683.0,5428977.0,1,https://maps.dnr.state.mn.us/airphotos/usda/byg/y1966/byg05gg123.jpg,{7E07CF60-6D46-4F27-8107-A47602220FB5},POINT (329682.221 5428977.671),usda_byg_y1966_byg05gg123.jpg,usda_byg_y1966_byg05gg123_georef.tif


In [18]:
filtered = gdf.loc[(gdf['year']==1989) & (gdf['project']=='stl')]

filtered.value_counts(dropna=False)

filename                             project  year  scale  provider  image_url                                                                  download_filename                   output_filename                            OBJECTID_1  photo_id     status  xutm      yutm       ext  globalid                                geometry                      
/projects/stl/y1989/stl_029_120.jpg  stl      1989  NaN    projects  https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_029_120.jpg  projects_stl_y1989_stl_029_120.jpg  projects_stl_y1989_stl_029_120_georef.tif  137998      stl_029_120  1       561926.0  5172458.0  jpg  {CCB4EDBA-7BEA-425C-9303-5A2995B7E585}  POINT (561925.715 5172458.418)    1
/projects/stl/y1989/stl_029_121.jpg  stl      1989  NaN    projects  https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_029_121.jpg  projects_stl_y1989_stl_029_121.jpg  projects_stl_y1989_stl_029_121_georef.tif  137999      stl_029_121  1       562024.0  5170972.0  jpg  {0E3E4A

In [ ]:
project_counts = filtered['project'].value_counts(dropna=False)
print(project_counts)

print("-------")

provider_counts = filtered['provider'].value_counts(dropna=False)
print(provider_counts)

### Approach to calculating extents

pixel_size_on_film_m  = 0.0254 / dpi
ground_pixel_size_m   = pixel_size_on_film_m * scale
extent_width_meters   = width_pixels * ground_pixel_size_m
extent_height_meters  = height_pixels * ground_pixel_size_m

Then send to magic georef: 
extent_meters = extent_width_meters  # Magic Georeferencer prefers horizontal
extent_dimension = "horizontal"

#### Next steps

1. Choose a working subset. I think I want to start with 3 photos each from 10 different years, all in St. Louis county. Save this in memory as gdf_test1
2. 